# Colab: `new_dataset` / `new_datamodule` unittest

**unittest만** 돌릴 때는 아래 3개 셀만 순서대로 실행하세요.

| 셀 | 내용 |
|----|------|
| 1 | repo clone / `git pull` + pip install |
| 2 | 경로 설정 |
| 3 | `tests/datamodules/test_new_*.py` 실행 |

> dev set · Semantic Hearing · SpAudSyn **불필요** (mock + temp WAV 사용)

Repo: https://github.com/Mockdd/dcase2026_task4

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Mockdd/dcase2026_task4.git"
COLAB_REPO_DIR = Path("/content/dcase2026_task4")

if IN_COLAB:
    if (COLAB_REPO_DIR / ".git").exists():
        print("[git pull] updating repo ...")
        subprocess.run(["git", "pull"], cwd=COLAB_REPO_DIR, check=True)
    elif not (COLAB_REPO_DIR / "src" / "datamodules" / "new_dataset.py").exists():
        print("[git clone]", REPO_URL)
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    else:
        print("[skip] repo already present (no .git — upload/clone manually)")

    %cd {COLAB_REPO_DIR}
    !pip install -q lightning pyyaml soundfile librosa scipy python-sofa tqdm

    !git log -1 --oneline
    print("\nColab setup OK:", COLAB_REPO_DIR)
else:
    print("Not Colab — run from local repo with: python -m unittest discover -s tests/datamodules -p test_new_*.py -v")

In [ ]:
import os
import sys
from pathlib import Path

import torch

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/dcase2026_task4")
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src" / "datamodules" / "new_dataset.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "tests" / "datamodules" / "test_new_dataset.py").exists(), \
    f"tests not found under {PROJECT_ROOT} — run cell 1 first"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

In [ ]:
import unittest
from io import StringIO


def run_datamodule_tests(verbosity=2):
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    suite.addTests(loader.discover(
        start_dir=str(PROJECT_ROOT / "tests" / "datamodules"),
        pattern="test_new_*.py",
        top_level_dir=str(PROJECT_ROOT),
    ))
    stream = StringIO()
    runner = unittest.TextTestRunner(stream=stream, verbosity=verbosity)
    result = runner.run(suite)
    print(stream.getvalue())
    print(f"\nRan {result.testsRun} tests | failures={len(result.failures)} | errors={len(result.errors)}")
    if result.failures:
        print("\n--- FAILURES ---")
        for test, tb in result.failures:
            print(test, "\n", tb)
    if result.errors:
        print("\n--- ERRORS ---")
        for test, tb in result.errors:
            print(test, "\n", tb)
    return result


result = run_datamodule_tests()
assert not result.failures and not result.errors, "unittest failed"
print("\nOK — new_dataset.py / new_datamodule.py unittest passed (26 tests)")